# Module 7 · Demo — Multi-Agent (Supervisor)

**From 0 to Agentic AI — DataHack Summit 2026**

One agent with many tools can get confused — too many jobs, one prompt. **Multi-agent** systems
split the work: each agent has a **narrow role, few tools, one job**. Here we build the most
common shape — a **supervisor** that routes to specialists.

> This is a **standalone teaching demo**. Our Knowledge Assistant stays a *single* agent — one
> good agent beats a tangle of mediocre ones. You reach for multi-agent only when a single agent
> genuinely can't hold the job.

### What you'll build
- Two **specialist** agents: a **researcher** (web) and a **docs expert** (internal knowledge)
- A **supervisor** that reads the question and routes to the right specialist
- A LangGraph where specialists report back to the supervisor until the task is done

---
## Setup

In [ ]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml.
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" "langgraph>=1.0,<2"

In [ ]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · Two specialist agents

Each specialist is just a small agent (from Module 2's `create_agent`) with **one** tool. Narrow
scope = reliable behaviour. (Canned tool data keeps the demo self-contained.)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)

@tool
def web_search(query: str) -> str:
    """Search the public web for current, external info."""
    return "LangGraph 1.0 was released in October 2025."

@tool
def company_docs(query: str) -> str:
    """Search internal company docs (HR, ops, finance)."""
    return "PTO is 28 days/year. On-call lead for billing is Sam."

researcher  = create_agent(llm, tools=[web_search],  system_prompt="You research the public web.")
docs_expert = create_agent(llm, tools=[company_docs], system_prompt="You answer from internal company docs.")

---
## Step 2 · Shared state &amp; the supervisor

The supervisor is an LLM that picks **who acts next** — a structured decision (`researcher`,
`docs_expert`, or `FINISH`). This is the coordination brain.

In [ ]:
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str

class Route(BaseModel):
    """Who should act next?"""
    next: Literal["researcher", "docs_expert", "FINISH"]

SUPERVISOR_PROMPT = (
    "You manage two specialists. Route to 'docs_expert' for internal company questions "
    "(HR, ops, people), and 'researcher' for general or current external info. "
    "When the last specialist has answered the question, reply FINISH."
)

def supervisor(state: State):
    decision = llm.with_structured_output(Route).invoke(
        [("system", SUPERVISOR_PROMPT)] + state["messages"])
    return {"next": decision.next}

---
## Step 3 · Wrap specialists as nodes

Each specialist runs, then its answer goes back into shared state **tagged with its name**, so
the supervisor (and we) can see who said what.

In [ ]:
from langchain_core.messages import HumanMessage

def make_node(agent, name):
    def node(state: State):
        result = agent.invoke({"messages": state["messages"]})
        answer = result["messages"][-1].content
        return {"messages": [HumanMessage(content=answer, name=name)]}
    return node

---
## Step 4 · Wire the graph

`START → supervisor`, a **conditional edge** to whichever specialist (or `END`), and each
specialist loops **back to the supervisor** — which decides again, until `FINISH`.

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

g = StateGraph(State)
g.add_node("supervisor", supervisor)
g.add_node("researcher", make_node(researcher, "researcher"))
g.add_node("docs_expert", make_node(docs_expert, "docs_expert"))
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda s: s["next"],
    {"researcher": "researcher", "docs_expert": "docs_expert", "FINISH": END})
g.add_edge("researcher", "supervisor")
g.add_edge("docs_expert", "supervisor")
team = g.compile()
display(Image(team.get_graph().draw_mermaid_png()))

---
## Step 5 · Run it — watch the routing

An internal question should go to the **docs expert**; an external one to the **researcher** —
the supervisor decides. (`recursion_limit` is our termination backstop from Module 4.)

In [ ]:
def ask(q):
    out = team.invoke({"messages": [("user", q)]}, {"recursion_limit": 8})
    return out["messages"][-1].content

print(ask("How many PTO days do I get?"))

In [ ]:
print(ask("What is the latest major version of LangGraph?"))

### See who did what
The tagged messages show the supervisor → specialist → supervisor flow.

In [ ]:
out = team.invoke({"messages": [("user", "Who is on-call for billing?")]}, {"recursion_limit": 8})
for m in out["messages"]:
    who = getattr(m, "name", None) or m.type
    print(f"[{who}] {m.content[:80]}")

---
## When NOT to do this

Multi-agent adds moving parts, latency, and ways to fail. The rule of thumb:
- ✅ **Split** when one agent juggles too many tools/goals and gets confused
- 🛑 **Don't** split a job a single agent already handles well — like our Knowledge Assistant

Start with one agent; grow into a team only when the task demands it.

---
## Key takeaways
- **Supervisor pattern:** a router agent delegates to narrow specialists, who report back.
- Coordination is just **shared state + a routing decision** — the same LangGraph primitives.
- **Swarm** is the alternative (agents hand off *peer-to-peer*, no central router) — see the slides.
- More agents = more complexity: split only when a single agent can't cope.

➡️ **Next (Module 8):** wrap our (single-agent) assistant in a **Streamlit** chat UI.